# 한 종류 학습 (Colab, L4)

공개된 Laya 체크포인트 두 개(`base`, `typed`) 중 validation 제로샷 정확도가 높은 쪽에서 한 종류를 파인튜닝하고, 학습 전후 validation 을 채점합니다. calibration split 이 있으면 그것으로 온도를 맞춰 확신도를 보정합니다. 유료 호출은 없고, 게이트 테스트는 읽지 않습니다.

## 준비 (PC)

    python laya/train.py pack useful

`laya/data/useful/colab-useful.zip` 을 Google Drive 의 `내 드라이브/laya/` 에 올립니다.

## 실행

1. 런타임 → 런타임 유형 변경 → L4 GPU
2. 런타임 → 모두 실행 (Drive 연결 권한은 허용)

## 결과

단계마다 `내 드라이브/laya/result-<종류>-<시각>.zip` 을 새로 저장합니다. 중간에 실패해도 그때까지의 로그와 오류가 들어 있습니다. 이 zip 을 PC 의 `laya/data/<종류>/` 에 내려받아 Claude 에게 넘기면 됩니다. 담기는 것:

- `log.txt` — 모든 셀의 출력과 오류
- `results.json` — 환경, 데이터, 실행마다 설정, 모델마다 정확도·평균 확신도·ECE·로그 손실, 선택된 모델, 단계별 소요 시간
- `predictions.jsonl` — validation 질문마다 정답과 각 모델의 답·확신도 (상태 본문은 없음)
- `<실행>-laya-run.json`, `<실행>-train_report.json`, `<실행>-calibration.json` — 학습 설정, Laya 의 학습 기록, 맞춘 온도

선택된 체크포인트만 크기가 커서(약 1.6 GB) 마지막 셀이 `내 드라이브/laya/<종류>-<split>-<실행>.zip` 에 따로 저장합니다. Claude 에게는 결과 zip 만 넘기면 됩니다.

개선 루프(`laya/loop.py`)는 유료 라벨링을 하므로 여기가 아니라 PC 에서 돕니다.

In [ ]:
import contextlib
import json
import logging
import shutil
import sys
import time
import traceback
import zipfile
from pathlib import Path

from google.colab import drive

KIND = "useful"
DRIVE = Path("/content/drive/MyDrive/laya")
STAMP = time.strftime("%Y%m%d-%H%M")
RESULT = Path(f"/content/result-{KIND}-{STAMP}")   # 이 폴더가 통째로 결과 zip 이 된다
RESULT.mkdir(exist_ok=True)
LOG = RESULT / "log.txt"
SUMMARY = {"kind": KIND, "started": STAMP, "seconds": {}}


class Tee:
    """화면에 쓰는 것을 로그 파일에도 쓴다. 나머지 속성(isatty 등)은 화면 쪽 스트림의 것을 쓴다."""

    def __init__(self, screen, file):
        self.screen, self.file = screen, file

    def write(self, text):
        self.file.write(text)
        return self.screen.write(text)

    def flush(self):
        self.file.flush()
        self.screen.flush()

    def __getattr__(self, name):
        return getattr(self.screen, name)


def save():
    """지금까지의 결과를 Drive 의 결과 zip 으로 다시 쓴다(같은 실행이면 같은 이름)."""
    (RESULT / "results.json").write_text(json.dumps(SUMMARY, indent=1, default=str), encoding="utf-8")
    shutil.make_archive(str(DRIVE / RESULT.name), "zip", RESULT)


@contextlib.contextmanager
def logged(step):
    """한 단계: 출력과 오류를 log.txt 에 남기고, 성공하든 실패하든 끝에 결과 zip 을 저장한다."""
    started = time.time()
    with open(LOG, "a", encoding="utf-8") as f:
        handler = logging.StreamHandler(f)   # Laya 가 logging 으로 남기는 것도 받는다
        logging.getLogger().addHandler(handler)
        f.write(f"\n===== {step} ({time.strftime('%H:%M:%S')}) =====\n")
        try:
            with contextlib.redirect_stdout(Tee(sys.stdout, f)), contextlib.redirect_stderr(Tee(sys.stderr, f)):
                try:
                    yield
                except BaseException:
                    traceback.print_exc()
                    SUMMARY["failed"] = step
                    raise
        finally:
            logging.getLogger().removeHandler(handler)
            SUMMARY["seconds"][step] = round(time.time() - started, 1)
            f.flush()   # zip 에 이 단계의 로그(실패라면 traceback)까지 들어가도록
            save()


drive.mount("/content/drive")
if not (DRIVE / f"colab-{KIND}.zip").exists():
    raise FileNotFoundError(f"{DRIVE}/colab-{KIND}.zip 이 없습니다. PC 에서 만든 zip 을 내 드라이브/laya/ 에 올려 주세요.")
%pip install -q laya==0.4.6 transformers==5.19.0

with logged("setup"):
    # 같은 런타임에서 다시 돌리면 앞 실행의 체크포인트 폴더가 남아 이름이 겹친다(앞 실행이 고른 것은 이미 Drive 에 있다)
    shutil.rmtree("/content/laya/checkpoints", ignore_errors=True)
    zipfile.ZipFile(DRIVE / f"colab-{KIND}.zip").extractall("/content")
    import importlib.metadata as md

    import torch

    SUMMARY["packed"] = json.loads(Path("/content/laya/packed.json").read_text(encoding="utf-8"))
    SUMMARY["env"] = {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                      "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
                      if torch.cuda.is_available() else None,
                      "python": sys.version.split()[0],
                      **{p: md.version(p) for p in ("laya", "transformers", "torch", "huggingface_hub")}}
    print(json.dumps(SUMMARY, indent=1))

In [ ]:
# 압축을 푼 laya/ 의 모듈을 이름으로 불러온다. `import laya` 는 설치된 패키지 그대로다.
sys.path.insert(0, "/content/laya")

import label  # noqa: E402
import train  # noqa: E402

with logged("data"):
    SUMMARY["data"] = {}
    for split in train.SPLITS:
        path = label.DATA / KIND / f"{split}.jsonl"
        if path.exists():
            rows = label.read(path)
            SUMMARY["data"][split] = {"states": len(rows), "questions": sum(len(r["expected"]) for r in rows)}
    VALIDATION = label.read(label.DATA / KIND / "validation.jsonl")
    print(SUMMARY["data"])

## 제로샷

공개 체크포인트 두 개를 validation 에서 채점합니다. 정확도가 높은 쪽이 파인튜닝의 출발점입니다(계획 4단계). 길이 상한(`train.MAX_LEN`)을 넘는 질문은 자르지 않고 채점에서 뺍니다.

In [ ]:
import math

ANSWERS, SURE = {}, {}   # 모델 이름 -> {(state_hash, 질문): 답}, {(state_hash, 질문): 확신도}


def ece(correct, confidence, bins=10):
    """확신도 구간마다 |정확도 - 평균 확신도| 를 질문 수로 가중 평균한 것."""
    total = 0.0
    for b in range(bins):
        inside = [i for i, c in enumerate(confidence) if b / bins < c <= (b + 1) / bins or (b == 0 and c == 0)]
        if inside:
            hit = sum(correct[i] for i in inside) / len(inside)
            total += len(inside) * abs(hit - sum(confidence[i] for i in inside) / len(inside))
    return total / len(confidence)


def log_loss(correct, confidence):
    """정답에 준 확률의 평균 -log. 맞으면 확신도, 틀리면 1 - 확신도가 정답의 확률이다(예/아니오 질문에서 정확)."""
    eps = 1e-6
    return sum(-math.log(min(max(c if ok else 1 - c, eps), 1 - eps)) for ok, c in zip(correct, confidence)) / len(correct)


def record(name, checkpoint):
    """한 모델을 채점해 정확도·확신도·ECE·로그 손실을 요약에 넣고, 질문별 답을 predictions.jsonl 로 다시 쓴다."""
    sure = {}
    wrong, predicted = train.score(KIND, checkpoint, sure)
    ANSWERS[name], SURE[name] = predicted, sure
    keys = list(predicted)
    correct = [k not in wrong for k in keys]
    confidence = [sure[k] for k in keys]
    SUMMARY.setdefault("accuracy", {})[name] = {
        "accuracy": round(sum(correct) / len(keys), 4), "wrong": len(wrong), "scored": len(keys),
        "mean_confidence": round(sum(confidence) / len(keys), 4), "ece": round(ece(correct, confidence), 4),
        "log_loss": round(log_loss(correct, confidence), 4),
        "mean_confidence_when_wrong": round(sum(c for c, ok in zip(confidence, correct) if not ok)
                                            / max(1, len(wrong)), 4)}
    with open(RESULT / "predictions.jsonl", "w", encoding="utf-8") as f:
        for row in VALIDATION:
            for q, want in row["expected"].items():
                key = (row["provenance"]["state_hash"], q)
                f.write(json.dumps({"state_hash": key[0], "question": q, "expected": want,
                                    **{n: a.get(key) for n, a in ANSWERS.items()},
                                    **{f"{n}_confidence": s.get(key) for n, s in SURE.items()}}) + "\n")
    print(name, SUMMARY["accuracy"][name])


with logged("zero-shot"):
    for name in train.PUBLISHED:
        record(name, name)
    BASE = max(train.PUBLISHED, key=lambda n: SUMMARY["accuracy"][n]["accuracy"])
    SUMMARY["base"] = BASE
    print("출발점:", BASE)

## 파인튜닝

fp16 과 gradient checkpointing 은 Laya 의 CUDA 기본값입니다. micro-batch 4 × 누적 16 으로 유효 배치 64 를 L4 의 24 GB 안에 맞춥니다. 옵션 순서와 무관하게 학습하도록 `option_layout="parallel"` 을 씁니다(`train.train` 의 기본).

`RUNS` 의 실행마다 한 번씩 학습하고 validation 으로 고릅니다. 실행은 이름과 `TrainConfig` 값(`epochs`, `seed`, `label_smoothing`, `encoder_lr` 등)이고, `train` 을 주면 train 에서 그 수만큼만 무작위로(시드 0) 골라 씁니다(학습 곡선). Laya 0.4.6 에는 early stopping 이 없어서 에폭 수도 실행으로 나눕니다. calibration split 이 있으면 학습 뒤 그것으로 온도를 맞추고, 채점은 그 온도를 적용한 확신도로 합니다. 로그 손실이 가장 낮은 것이 선택됩니다. 이 모델은 확신할 때만 답하고 나머지는 미루므로(임계값 정책), 정확도만이 아니라 확신도가 맞고 틀림을 얼마나 가르는지도 봐야 합니다. 300문항 실행에서 4에폭은 정확도가 0.78 로 2에폭(0.74)보다 높았지만, 과신이 커서 온도가 상한 5.0 에 막혔고 확신도 0.9 이상에서도 정확도가 0.83 에 그쳤습니다. 2에폭은 확신도 0.8 이상 113문항에서 0.94 였습니다.

지금 `RUNS` 는 2에폭(실행 4 의 선택)을 기준으로 한 비교입니다: 시드 차이(같은 설정을 시드 1, 2 로), 학습 곡선(train 250), label smoothing 0.1, encoder 학습률 1.5e-5. 실행 하나가 약 3분입니다.

In [ ]:
RUNS = [
    {"name": "e2", "epochs": 2},                                   # 실행 4 의 선택, 기준
    {"name": "e2-seed1", "epochs": 2, "seed": 1},                  # 시드만 바꾼 차이
    {"name": "e2-seed2", "epochs": 2, "seed": 2},
    {"name": "e2-train250", "epochs": 2, "train": 250},            # 학습 곡선: 데이터를 늘리면 나아지는가
    {"name": "e2-ls0.1", "epochs": 2, "label_smoothing": 0.1},     # 과신 억제
    {"name": "e3-ls0.1", "epochs": 3, "label_smoothing": 0.1},
    {"name": "e2-lr1.5", "epochs": 2, "encoder_lr": 1.5e-5},       # 작은 학습률(기본 2.5e-5)
    {"name": "e3-lr1.5", "epochs": 3, "encoder_lr": 1.5e-5},
]
CHECKPOINTS = {}   # 모델 이름 -> 체크포인트 폴더
SUMMARY["runs"] = {}

for run in RUNS:
    config = dict(run)
    short, limit = config.pop("name"), config.pop("train", None)
    name = f"fine-tuned-{short}"
    with logged(f"fine-tune {short}"):
        split, rows = "train", SUMMARY["data"]["train"]["states"]
        if limit:
            # train 에서 limit 개 레코드를 시드 0 으로 골라 별도 split 으로 쓴다
            split, rows = train.subset(KIND, limit)
        done = train.train(KIND, [split], base=BASE, micro_batch=4, grad_accum=16, **config)
        # 다음 학습이 같은 폴더에 쓰지 않도록 실행 이름을 붙여 옮긴다
        CHECKPOINTS[name] = done.rename(done.with_name(f"{done.name}-{short}"))
        SUMMARY["runs"][name] = {**config, "train": rows}
        for file in ("laya-run.json", "train_report.json", "calibration.json"):
            if (CHECKPOINTS[name] / file).exists():
                shutil.copy(CHECKPOINTS[name] / file, RESULT / f"{short}-{file}")
        if torch.cuda.is_available():
            SUMMARY["env"][f"peak_gpu_memory_gb_{short}"] = round(torch.cuda.max_memory_allocated() / 2**30, 1)
            torch.cuda.reset_peak_memory_stats()
    with logged(f"score {short}"):
        record(name, CHECKPOINTS[name])

BEST = min(CHECKPOINTS, key=lambda n: SUMMARY["accuracy"][n]["log_loss"])
SUMMARY["chosen"] = BEST
print("선택:", BEST, SUMMARY["accuracy"][BEST])

In [ ]:
# 선택된 체크포인트(약 1.6 GB)만 결과 zip 과 따로 저장한다. 결과 zip 은 위 단계마다 이미 저장됐다.
with logged("checkpoint"):
    SUMMARY["checkpoint_zip"] = shutil.make_archive(str(DRIVE / CHECKPOINTS[BEST].name), "zip", CHECKPOINTS[BEST])
print("결과 zip:", DRIVE / f"{RESULT.name}.zip")
print("체크포인트 zip:", SUMMARY["checkpoint_zip"])